<a href="https://colab.research.google.com/github/replysantosh-lang/ECARDeepLearning/blob/main/DAY03/Fine_Tuning_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Transfer Learning + Fine Tuning using VGG16

from tensorflow.keras.applications import VGG16
from tensorflow.keras import models, layers, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Load pretrained VGG16 convolution base
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150,150,3)
)

# Build model
model = models.Sequential()
model.add(conv_base)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

# Freeze convolution base
print("Trainable weights before freezing:", len(model.trainable_weights))
conv_base.trainable = False
print("Trainable weights after freezing:", len(model.trainable_weights))


# Dataset directories
#train_dir="/content/cats_dogs_dataset/train/"
#validation_dir="/content/cats_dogs_dataset/validation/"

train_dir="/content/drive/MyDrive/cats_dogs_dataset/cats_dogs_dataset/train/"
validation_dir="/content/drive/MyDrive/cats_dogs_dataset/cats_dogs_dataset/validation/"

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(rescale=1./255)

# Generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

# Compile model (initial training)
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=2e-5),
    metrics=['accuracy']
)

# Train top classifier
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=30,
    validation_data=validation_generator,
    validation_steps=50
)

# ------------------------
# Fine Tuning
# ------------------------

conv_base.trainable = True

set_trainable = False
for layer in conv_base.layers:
    if layer.name == 'block5_conv1':
        set_trainable = True
    layer.trainable = set_trainable

# Recompile with lower learning rate
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-5),
    metrics=['accuracy']
)

# Fine-tune training
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=100,
    validation_data=validation_generator,
    validation_steps=50
)

# Plot results
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc)+1)

plt.plot(epochs, acc, 'bo', label='Training Accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.figure()

plt.plot(epochs, loss, 'bo', label='Training Loss')
plt.plot(epochs, val_loss, 'b', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.show()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Trainable weights before freezing: 30
Trainable weights after freezing: 4
Found 3598 images belonging to 2 classes.
Found 400 images belonging to 2 classes.
Epoch 1/30
 86/100 ━━━━━━━━━━━━━━━━━━━━ 1:48 8s/step - accuracy: 0.5733 - loss: 0.6654

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5862 - loss: 0.6572

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


100/100 ━━━━━━━━━━━━━━━━━━━━ 941s 9s/step - accuracy: 0.6735 - loss: 0.6010 - val_accuracy: 0.7875 - val_loss: 0.4580
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.7452 - loss: 0.5203 - val_accuracy: 0.8550 - val_loss: 0.4222
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 928s 9s/step - accuracy: 0.7932 - loss: 0.4701 - val_accuracy: 0.8675 - val_loss: 0.3538
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 192s 2s/step - accuracy: 0.7764 - loss: 0.4697 - val_accuracy: 0.8825 - val_loss: 0.3370
Epoch 5/30
 87/100 ━━━━━━━━━━━━━━━━━━━━ 1:38 8s/step - accuracy: 0.8035 - loss: 0.4380

KeyboardInterrupt: 